In [ ]:
import numpy as np
import pandas as pd
import pickle
from scipy.optimize import differential_evolution
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load models
with open('../data/processed/tuned_secom_model.pkl', 'rb') as f:
    yield_model = pickle.load(f)

with open('../data/processed/best_steel_model.pkl', 'rb') as f:
    energy_model = pickle.load(f)

X_steel = pd.read_csv('../data/processed/X_steel.csv')

print("Models loaded!")
print(f"Steel feature columns: {list(X_steel.columns)}")

In [ ]:
# ===== Multi-Objective Optimization =====

bounds = [(X_steel[col].min(), X_steel[col].max()) for col in X_steel.columns]

def objective(params):
    x = np.array(params).reshape(1, -1)
    df = pd.DataFrame(x, columns=X_steel.columns)
    energy = energy_model.predict(df)[0]
    energy_score = energy / 100
    return energy_score

print("Generating Pareto front solutions...")
pareto_solutions = []

for load_type in [0, 1, 2]:
    for hour in range(0, 24, 4):
        sample = X_steel.mean().copy()
        sample['Load_Type_encoded'] = load_type
        sample['hour'] = hour
        sample['is_peak_hour'] = 1 if 8 <= hour <= 18 else 0
        df = pd.DataFrame([sample])
        energy = energy_model.predict(df)[0]
        pareto_solutions.append({
            'hour': hour,
            'load_type': ['Light', 'Medium', 'Maximum'][load_type],
            'predicted_energy_kwh': round(energy, 2),
            'is_peak': 'Peak' if 8 <= hour <= 18 else 'Off-Peak'
        })

pareto_df = pd.DataFrame(pareto_solutions)
print(pareto_df.sort_values('predicted_energy_kwh').head(10))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'Light': '#2ecc71', 'Medium': '#f39c12', 'Maximum': '#e74c3c'}
for load in ['Light', 'Medium', 'Maximum']:
    subset = pareto_df[pareto_df['load_type'] == load]
    axes[0].scatter(subset['hour'], subset['predicted_energy_kwh'],
                   label=load, color=colors[load], s=80, alpha=0.8)

axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Predicted Energy (kWh)')
axes[0].set_title('Energy Consumption by Hour and Load Type')
axes[0].legend()
axes[0].axvspan(8, 18, alpha=0.1, color='yellow')

best = pareto_df.nsmallest(5, 'predicted_energy_kwh')
worst = pareto_df.nlargest(5, 'predicted_energy_kwh')
compare = pd.concat([best, worst])
compare['label'] = compare['load_type'] + ' H:' + compare['hour'].astype(str)
bar_colors = ['#2ecc71'] * 5 + ['#e74c3c'] * 5
axes[1].barh(compare['label'], compare['predicted_energy_kwh'], color=bar_colors, edgecolor='white')
axes[1].set_xlabel('Predicted Energy (kWh)')
axes[1].set_title('Best vs Worst Operating Conditions')
axes[1].axvline(x=pareto_df['predicted_energy_kwh'].mean(), color='black', linestyle='--', label='Average')
axes[1].legend()

plt.tight_layout()
plt.savefig('../reports/figures/pareto_optimization.png', dpi=150, bbox_inches='tight')
plt.show()